In [1]:
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\Text_Summariser\\reseach'

In [2]:
import os
os.chdir("../")
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\Text_Summariser'

In [3]:
# define entities
# from config.yaml
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: str

In [4]:
from Text_summariser.constant import *
from Text_summariser.utils.common import read_yaml, create_directories

In [5]:
# 4 Update configuration manager

class configurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root]) # same upto here for most pipeline

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir = config.root_dir,
            data_path = config.data_path,
            tokenizer_name = config.tokenizer_name
        )

        return data_transformation_config

In [6]:
import os
from Text_summariser.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

c:\Users\Sandeep\anaconda3\envs\textS\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# Conponents
class DataTransformation:
    def __init__(self, config):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)


    def convert_examples_to_features(self, examples):
        inputs = ["summarize: " + doc for doc in examples["dialogue"]]

        model_inputs = self.tokenizer(
            inputs,
            max_length=512,
            truncation=True,
            padding="max_length"
        )

        labels = self.tokenizer(
            text_target=examples["summary"],
            max_length=128,
            truncation=True,
            padding="max_length"
        )

        model_inputs["labels"] = labels["input_ids"]

        return {
            "input_ids": model_inputs["input_ids"],
            "attention_mask": model_inputs["attention_mask"],
            "labels": labels["input_ids"]
        }
    
    def convert(self):
        # Load dataset from disk
        dataset_samsum = load_from_disk(self.config.data_path)

        # Tokenize dataset
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched=True)

        # Save tokenized dataset
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir, "samsum_dataset"))

In [8]:
# Pipeline
try:
    config = configurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config = data_transformation_config)
    data_transformation.convert()

except Exception as e:
    raise e

[2026-05-15 15:28:55,393: INFO: common: ymal file config\config.yaml loaded sucessfully]
[2026-05-15 15:28:55,397: INFO: common: ymal file params.yaml loaded sucessfully]
[2026-05-15 15:28:55,399: INFO: common: created directory at artifacts]
[2026-05-15 15:28:55,400: INFO: common: created directory at artifacts/data_transformation]


c:\Users\Sandeep\anaconda3\envs\textS\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 67600.75 examples/s]
